In [14]:
import os
import sys
import torch
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

In [15]:
# 벡터 데이터베이스 로드
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vector_db = Chroma(persist_directory="./chroma_db_session", embedding_function=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k":2})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
local_model_dir = './local_qwen_model'

In [22]:
if not os.path.exists(local_model_dir):
    os.makedirs(local_model_dir, exist_ok=True)
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(model_id,torch_dtype = torch.float32)
    _tokenizer.save_pretrained(local_model_dir)
    _model.save_pretrained(local_model_dir)

tokenizer = AutoTokenizer.from_pretrained(local_model_dir)
model = AutoModelForCausalLM.from_pretrained(local_model_dir, torch_dtype=torch.float32,
                                             device_map='cpu', low_cpu_mem_usage=True)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [23]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    max_length=None,
    temperature=0.1,
    do_sample=True,
    clean_up_tokenization_spaces=False,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=pipe)

In [24]:
class GraphState(TypedDict):
    question:str
    context:str
    generation:str
    source_info:str

def retrieve(state:GraphState):
    print('--retrieve node--')
    question = state['question']
    docs = retriever.invoke(question)
    contexts, sources = [], []

    for i, doc in enumerate(docs):
        # 텍스트 데이터 포맷팅
        contexts.append(f'[문서 {i+1}] {doc.page_content}')

        # 메타 데이터 추출
        filename = doc.metadata.get('file_name', 'N/A')
        f_type = doc.metadata.get('file_type', 'N/A')
        cat = doc.metadata.get('category', 'N/A')
        sources.append(f'문서 {i+1} [파일명: {filename} / 유형: {f_type} / 카테고리: {cat}]')

    contexts_str = '\n\n'.join(contexts)
    sources_str = '\n\n'.join(sources)
    print(f'Retrieve {len(docs)} documents')
    
    return {"context": contexts_str, "source_info":sources_str}

def generate(state:GraphState):
    print("-- generate node --")
    question = state['question']
    context = state['context']

    # LLM 을 위한 프롬프트 템플릿
    messages = [
        {'role':'system', 'content':'주어진 문장을 참고해서 사용자의 질문에 한국어로 정확하고 간결하게 답변하세요.'},
        {'role':'user', 'content':f'[본문]\n{context}\n\n[질문]\n{question}'}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = llm.invoke(prompt)
    return {'generation':response.strip(), "source_info":state["source_info"]}

In [25]:
workflow = StateGraph(GraphState)
workflow.add_node('retrieve', retrieve)
workflow.add_node('generate', generate)
workflow.set_entry_point('retrieve')
workflow.add_edge('retrieve', 'generate')
workflow.add_edge('generate', END)
app = workflow.compile()

In [26]:
user_question = input("질문\n")
inputs = {'question':user_question}
final_state = app.invoke(inputs)
print('\n==================답변==================')
print(final_state['generation'])
print("\n==================출처 정보====================")
print(final_state['source_info'])
print("\n==================================================")

--retrieve node--
Retrieve 0 documents
-- generate node --

==================답변==================
RAG (Revised AI Generation) 시스템은 기존의 AI 시스템과는 다르게, 데이터를 직접적으로 학습하거나 인공지능을 통해 생성하는 대신, 인간의 지식이나 경험을 바탕으로 자동화된 정보 생성을 수행합니다. 이 시스템은 문서 검색, 질문 응답, 또는 특정 주제에 대한 내용 생성 등 다양한 작업에서 활용될 수 있습니다.

==================출처 정보====================




-> 이렇게 하면 질문을 한번 밖에 못함.
01.py 파일로 여러번 질문할 수 있도록 코드 수정해보자!!